In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()

TRAIN_DIR = PROJECT_ROOT / "student_resource" / "dataset" / "train"

SRC_DIR = (
    PROJECT_ROOT
    / "code"
    / "business_entity_resolution"
    / "src"
)

sys.path.insert(0, str(SRC_DIR))

from business_entity_resolution.preprocessing.normalization import (
    normalize_name,
    normalize_address
)

In [2]:
s1 = pd.read_csv(
    TRAIN_DIR / "train_source1.tsv",
    sep="\t",
    usecols=[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
)

s2 = pd.read_csv(
    TRAIN_DIR / "train_source2.tsv",
    sep="\t",
    usecols=[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
)

s3 = pd.read_csv(
    TRAIN_DIR / "train_source3.tsv",
    sep="\t",
    usecols=[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
)

In [3]:
for df in [s1, s2, s3]:

    df["name_norm"] = (
        df["business_name"]
        .map(normalize_name)
    )

In [4]:
s1["block_key"] = (
    s1["country"].fillna("").astype(str)
    + "||"
    + s1["name_norm"]
)

s2["block_key"] = (
    s2["country"].fillna("").astype(str)
    + "||"
    + s2["name_norm"]
)

s3["block_key"] = (
    s3["country"].fillna("").astype(str)
    + "||"
    + s3["name_norm"]
)

In [5]:
gt = pd.read_csv(
    TRAIN_DIR / "train_ground_truth.tsv",
    sep="\t"
)

gt["matched_entity_ids"] = (
    gt["matched_entity_ids"]
    .fillna("")
    .astype(str)
)

In [6]:
gt_positive = gt[
    gt["matched_entity_ids"].str.strip() != ""
].copy()

sample_gt = gt_positive.sample(
    n=min(10000, len(gt_positive)),
    random_state=42
)

In [7]:
positive_pairs = (
    sample_gt[
        ["source1_entity_id", "matched_entity_ids"]
    ]
    .assign(
        matched_entity_id=lambda x:
            x["matched_entity_ids"].str.split(",")
    )
    .explode("matched_entity_id")
)

positive_pairs["matched_entity_id"] = (
    positive_pairs["matched_entity_id"]
    .str.strip()
)

In [8]:
positive_pairs = positive_pairs.merge(
    s1[
        ["entity_id", "block_key"]
    ],
    left_on="source1_entity_id",
    right_on="entity_id",
    how="left"
)

positive_pairs = positive_pairs.rename(
    columns={
        "block_key": "s1_block_key"
    }
)

positive_pairs = positive_pairs.drop(
    columns=["entity_id"]
)

In [9]:
s23_keys = pd.concat(
    [
        s2[
            ["entity_id", "block_key"]
        ],
        s3[
            ["entity_id", "block_key"]
        ]
    ],
    ignore_index=True
)

In [10]:
positive_pairs = positive_pairs.merge(
    s23_keys,
    left_on="matched_entity_id",
    right_on="entity_id",
    how="left"
)

positive_pairs = positive_pairs.rename(
    columns={
        "block_key": "matched_block_key"
    }
)

positive_pairs = positive_pairs.drop(
    columns=["entity_id"]
)

In [11]:
positive_pairs["blocked"] = (
    positive_pairs["s1_block_key"]
    ==
    positive_pairs["matched_block_key"]
)

In [12]:
blocking_recall = (
    positive_pairs["blocked"].mean()
)

print(
    "Blocking recall:",
    round(blocking_recall * 100, 2),
    "%"
)

print(
    "Positive pairs:",
    len(positive_pairs)
)

print(
    "Captured:",
    positive_pairs["blocked"].sum()
)

Blocking recall: 24.04 %
Positive pairs: 36703
Captured: 8825


In [13]:
eval_s1_ids = positive_pairs["source1_entity_id"].unique()

eval_s2_ids = positive_pairs.loc[
    positive_pairs["matched_entity_id"].str.startswith("S2-"),
    "matched_entity_id"
].unique()

eval_s3_ids = positive_pairs.loc[
    positive_pairs["matched_entity_id"].str.startswith("S3-"),
    "matched_entity_id"
].unique()

print("S1:", len(eval_s1_ids))
print("S2:", len(eval_s2_ids))
print("S3:", len(eval_s3_ids))

S1: 10000
S2: 17704
S3: 18999


In [14]:
s1_addr_eval = s1[
    s1["entity_id"].isin(eval_s1_ids)
][
    ["entity_id", "business_address"]
].copy()

s2_addr_eval = s2[
    s2["entity_id"].isin(eval_s2_ids)
][
    ["entity_id", "business_address"]
].copy()

s3_addr_eval = s3[
    s3["entity_id"].isin(eval_s3_ids)
][
    ["entity_id", "business_address"]
].copy()

In [15]:
s1_addr_eval["address_norm"] = (
    s1_addr_eval["business_address"]
    .map(normalize_address)
)

s2_addr_eval["address_norm"] = (
    s2_addr_eval["business_address"]
    .map(normalize_address)
)

s3_addr_eval["address_norm"] = (
    s3_addr_eval["business_address"]
    .map(normalize_address)
)

In [16]:
s1_addr_eval["address_block"] = (
    s1_addr_eval["address_norm"]
)

s2_addr_eval["address_block"] = (
    s2_addr_eval["address_norm"]
)

s3_addr_eval["address_block"] = (
    s3_addr_eval["address_norm"]
)

In [17]:
s23_addr_eval = pd.concat(
    [
        s2_addr_eval[["entity_id", "address_block"]],
        s3_addr_eval[["entity_id", "address_block"]]
    ],
    ignore_index=True
)

In [18]:
positive_pairs = positive_pairs.drop(
    columns=["address_block_s1", "address_block_matched"],
    errors="ignore"
)
positive_pairs = positive_pairs.merge(
    s1_addr_eval[
        ["entity_id", "address_block"]
    ],
    left_on="source1_entity_id",
    right_on="entity_id",
    how="left"
)

positive_pairs = positive_pairs.rename(
    columns={
        "address_block": "address_block_s1"
    }
)

positive_pairs = positive_pairs.drop(
    columns=["entity_id"]
)
positive_pairs = positive_pairs.merge(
    s23_addr_eval,
    left_on="matched_entity_id",
    right_on="entity_id",
    how="left"
)

positive_pairs = positive_pairs.rename(
    columns={
        "address_block": "address_block_matched"
    }
)

positive_pairs = positive_pairs.drop(
    columns=["entity_id"]
)

In [19]:
positive_pairs["block_b"] = (
    positive_pairs["address_block_s1"]
    ==
    positive_pairs["address_block_matched"]
)

block_b_recall = positive_pairs["block_b"].mean()

print(
    "Block B recall:",
    round(block_b_recall * 100, 2),
    "%"
)

print(
    "Block B captured:",
    positive_pairs["block_b"].sum()
)

Block B recall: 11.67 %
Block B captured: 4285


In [20]:
positive_pairs["block_a_or_b"] = (
    positive_pairs["blocked"]
    |
    positive_pairs["block_b"]
)

In [21]:
union_recall = (
    positive_pairs["block_a_or_b"].mean()
)

print(
    "Block A recall:",
    round(
        positive_pairs["blocked"].mean() * 100,
        2
    ),
    "%"
)

print(
    "Block B recall:",
    round(
        positive_pairs["block_b"].mean() * 100,
        2
    ),
    "%"
)

print(
    "A OR B recall:",
    round(
        union_recall * 100,
        2
    ),
    "%"
)

print(
    "A OR B captured:",
    positive_pairs["block_a_or_b"].sum()
)

Block A recall: 24.04 %
Block B recall: 11.67 %
A OR B recall: 33.38 %
A OR B captured: 12250


In [22]:
import re
def name_tokens(text):
    if not text:
        return []

    tokens = str(text).split()
    return [token for token in tokens if len(token) >= 3]


NAME_STOPWORDS = {
    "private",
    "limited",
    "corporation",
    "company",
    "incorporated",
    "holding",
    "holdings",
    "group",
}


def informative_name_tokens(text):
    tokens = name_tokens(text)
    return [token for token in tokens if token not in NAME_STOPWORDS]


def token_overlap(a, b):
    a_tokens = set(informative_name_tokens(a))
    b_tokens = set(informative_name_tokens(b))

    if not a_tokens or not b_tokens:
        return False

    return bool(a_tokens & b_tokens)

In [23]:
print(positive_pairs.columns.tolist())

['source1_entity_id', 'matched_entity_ids', 'matched_entity_id', 's1_block_key', 'matched_block_key', 'blocked', 'address_block_s1', 'address_block_matched', 'block_b', 'block_a_or_b']


In [24]:
positive_pairs["name_token_overlap"] = positive_pairs.apply(
    lambda row: token_overlap(
        row["s1_block_key"],
        row["matched_block_key"]
    ),
    axis=1
)

token_recall = positive_pairs["name_token_overlap"].mean() * 100

print(f"Positive pairs sharing >=1 informative name token: {token_recall:.2f}%")

Positive pairs sharing >=1 informative name token: 82.97%


In [25]:
def extract_name_from_block(block):
    if not block:
        return ""

    parts = str(block).split("||", 1)

    if len(parts) == 2:
        return parts[1]

    return parts[0]

In [26]:
positive_pairs["s1_name_for_token"] = positive_pairs["s1_block_key"].apply(
    extract_name_from_block
)

positive_pairs["matched_name_for_token"] = positive_pairs["matched_block_key"].apply(
    extract_name_from_block
)

positive_pairs["name_token_overlap"] = positive_pairs.apply(
    lambda row: token_overlap(
        row["s1_name_for_token"],
        row["matched_name_for_token"]
    ),
    axis=1
)

token_recall = positive_pairs["name_token_overlap"].mean() * 100

print(f"Positive pairs sharing >=1 informative name token: {token_recall:.2f}%")

Positive pairs sharing >=1 informative name token: 84.31%


In [27]:
from collections import Counter

token_counter = Counter()

for text in positive_pairs["s1_name_for_token"]:
    token_counter.update(set(informative_name_tokens(text)))

for text in positive_pairs["matched_name_for_token"]:
    token_counter.update(set(informative_name_tokens(text)))

print("Unique tokens:", len(token_counter))

print("\nMost common tokens:")
for token, count in token_counter.most_common(30):
    print(f"{token:25} {count}")

Unique tokens: 16332

Most common tokens:
llc                       10238
and                       4974
center                    2016
com                       1968
services                  1722
india                     1527
care                      1313
partners                  1273
associates                1260
llp                       1052
trading                   878
health                    736
technologies              696
solutions                 695
international             634
clinic                    607
brothers                  602
global                    588
service                   574
medicine                  555
enterprises               533
pllc                      526
pediatric                 509
trust                     502
foundation                501
industries                460
capital                   448
products                  447
consultancy               442
consultants               434


In [28]:
def token_overlap_count(a, b):
    a_tokens = set(informative_name_tokens(a))
    b_tokens = set(informative_name_tokens(b))

    return len(a_tokens & b_tokens)


positive_pairs["shared_name_tokens"] = positive_pairs.apply(
    lambda row: token_overlap_count(
        row["s1_name_for_token"],
        row["matched_name_for_token"]
    ),
    axis=1
)

for n in [1, 2, 3]:
    recall = (
        positive_pairs["shared_name_tokens"] >= n
    ).mean() * 100

    print(f">= {n} shared informative token(s): {recall:.2f}%")

>= 1 shared informative token(s): 84.31%
>= 2 shared informative token(s): 69.68%
>= 3 shared informative token(s): 30.14%


In [29]:
NAME_STOPWORDS = {
    # Legal/business suffixes
    "private",
    "limited",
    "corporation",
    "company",
    "incorporated",
    "llc",
    "llp",
    "pllc",
    "corp",
    "inc",
    "co",

    # Generic business terms
    "group",
    "holding",
    "holdings",
    "services",
    "service",
    "solutions",
    "enterprises",
    "international",
    "global",
    "partners",
    "associates",
    "consultancy",
    "consultants",

    # Very generic words
    "and",
    "india",
}

In [30]:
def informative_name_tokens_v2(text):
    tokens = name_tokens(text)
    return [token for token in tokens if token not in NAME_STOPWORDS]


def token_overlap_v2(a, b):
    a_tokens = set(informative_name_tokens_v2(a))
    b_tokens = set(informative_name_tokens_v2(b))

    if not a_tokens or not b_tokens:
        return False

    return bool(a_tokens & b_tokens)


positive_pairs["name_token_overlap_v2"] = positive_pairs.apply(
    lambda row: token_overlap_v2(
        row["s1_name_for_token"],
        row["matched_name_for_token"]
    ),
    axis=1
)

recall_v2 = (
    positive_pairs["name_token_overlap_v2"].mean() * 100
)

print(f"Token blocking recall after removing generic tokens: {recall_v2:.2f}%")

Token blocking recall after removing generic tokens: 83.53%


In [31]:
def token_overlap_count_v2(a, b):
    a_tokens = set(informative_name_tokens_v2(a))
    b_tokens = set(informative_name_tokens_v2(b))

    return len(a_tokens & b_tokens)


positive_pairs["shared_tokens_v2"] = positive_pairs.apply(
    lambda row: token_overlap_count_v2(
        row["s1_name_for_token"],
        row["matched_name_for_token"]
    ),
    axis=1
)

for n in [1, 2, 3]:
    recall = (
        positive_pairs["shared_tokens_v2"] >= n
    ).mean() * 100

    print(f">= {n} informative token(s): {recall:.2f}%")

>= 1 informative token(s): 83.53%
>= 2 informative token(s): 61.76%
>= 3 informative token(s): 19.69%


In [32]:
print(s2.columns.tolist())
print(s3.columns.tolist())

['entity_id', 'business_name', 'business_address', 'country', 'name_norm', 'block_key']
['entity_id', 'business_name', 'business_address', 'country', 'name_norm', 'block_key']


In [33]:
from collections import Counter

NAME_STOPWORDS = {
    "private",
    "limited",
    "corporation",
    "company",
    "incorporated",
    "llc",
    "llp",
    "pllc",
    "corp",
    "inc",
    "co",
    "group",
    "holding",
    "holdings",
    "services",
    "service",
    "solutions",
    "enterprises",
    "international",
    "global",
    "partners",
    "associates",
    "consultancy",
    "consultants",
    "and",
    "india",
}


def informative_name_tokens_v2(text):
    if not text:
        return []

    tokens = str(text).split()

    return [
        token
        for token in tokens
        if len(token) >= 3 and token not in NAME_STOPWORDS
    ]

In [34]:
token_frequency = Counter()

for df in [s2, s3]:
    for name, country in zip(df["name_norm"], df["country"]):
        tokens = set(informative_name_tokens_v2(name))

        for token in tokens:
            token_frequency[(country, token)] += 1

In [35]:
print("Unique country-token blocks:", len(token_frequency))

print("\nMost frequent country-token blocks:")

for (country, token), count in token_frequency.most_common(30):
    print(f"{country:5} | {token:25} | {count:,}")

Unique country-token blocks: 1517366

Most frequent country-token blocks:
US    | center                    | 285,248
US    | com                       | 266,404
US    | care                      | 156,113
India | com                       | 146,176
India | center                    | 129,257
US    | health                    | 99,434
US    | the                       | 92,042
India | industries                | 81,588
US    | clinic                    | 75,402
India | ventures                  | 73,301
India | brothers                  | 70,450
India | public                    | 70,374
India | trading                   | 68,479
US    | pediatric                 | 67,983
US    | medicine                  | 64,468
India | exports                   | 63,403
India | sri                       | 55,060
India | shri                      | 54,919
India | smt                       | 54,762
US    | dental                    | 53,061
India | technologies              | 51,556
US    | family    

In [36]:
import numpy as np

frequencies = np.array(list(token_frequency.values()))

print("Min:", frequencies.min())
print("Median:", np.median(frequencies))
print("Mean:", frequencies.mean())
print("95th percentile:", np.percentile(frequencies, 95))
print("99th percentile:", np.percentile(frequencies, 99))
print("Max:", frequencies.max())

Min: 1
Median: 1.0
Mean: 15.17574995090176
95th percentile: 14.0
99th percentile: 122.0
Max: 285248


In [37]:
positive_pairs.columns.tolist()

['source1_entity_id',
 'matched_entity_ids',
 'matched_entity_id',
 's1_block_key',
 'matched_block_key',
 'blocked',
 'address_block_s1',
 'address_block_matched',
 'block_b',
 'block_a_or_b',
 'name_token_overlap',
 's1_name_for_token',
 'matched_name_for_token',
 'shared_name_tokens',
 'name_token_overlap_v2',
 'shared_tokens_v2']

In [38]:
frequency_thresholds = [10, 25, 50, 100, 250, 500, 1000]

def get_country_from_block(block):
    if not block:
        return ""

    return str(block).split("||", 1)[0]


def get_name_from_block(block):
    if not block:
        return ""

    parts = str(block).split("||", 1)

    if len(parts) == 2:
        return parts[1]

    return parts[0]


def get_shared_tokens_under_threshold(row, threshold):
    country = get_country_from_block(row["s1_block_key"])

    s1_name = get_name_from_block(row["s1_block_key"])
    matched_name = get_name_from_block(row["matched_block_key"])

    s1_tokens = set(informative_name_tokens_v2(s1_name))
    matched_tokens = set(informative_name_tokens_v2(matched_name))

    shared = s1_tokens & matched_tokens

    return any(
        token_frequency.get((country, token), 0) <= threshold
        for token in shared
    )


results = []

for threshold in frequency_thresholds:

    captured = positive_pairs.apply(
        lambda row: get_shared_tokens_under_threshold(
            row,
            threshold
        ),
        axis=1
    )

    recall = captured.mean() * 100

    results.append({
        "frequency_threshold": threshold,
        "positive_pairs_captured": int(captured.sum()),
        "recall_percent": recall
    })


frequency_results = pd.DataFrame(results)

frequency_results

,frequency_threshold,positive_pairs_captured,recall_percent
0,10,973,2.651009
1,25,2613,7.119309
2,50,4545,12.383184
3,100,6875,18.731439
4,250,12245,33.362395
5,500,14007,38.163093
6,1000,15365,41.863063


In [39]:
print(
    frequency_results.to_string(index=False)
)

 frequency_threshold  positive_pairs_captured  recall_percent
                  10                      973        2.651009
                  25                     2613        7.119309
                  50                     4545       12.383184
                 100                     6875       18.731439
                 250                    12245       33.362395
                 500                    14007       38.163093
                1000                    15365       41.863063


In [40]:
hard_token_cases = positive_pairs[
    positive_pairs["shared_tokens_v2"] == 0
]

print("Positive pairs with ZERO informative shared tokens:")
print(len(hard_token_cases))

print(
    "Percentage:",
    round(len(hard_token_cases) / len(positive_pairs) * 100, 2),
    "%"
)

Positive pairs with ZERO informative shared tokens:
6045
Percentage: 16.47 %


In [41]:
hard_token_cases[
    [
        "s1_name_for_token",
        "matched_name_for_token",
        "address_block_s1",
        "address_block_matched",
        "block_a_or_b"
    ]
].head(20)

,s1_name_for_token,matched_name_for_token,address_block_s1,address_block_matched,block_a_or_b
4,ms consultancy corporation,ms consultancy,shymala appts 1st floor flat no 4 opp ratna ho...,shymala appts 1 1st floor flat no 4 opp ratna ...,False
5,ms consultancy corporation,m5 consultancy corporation,shymala appts 1st floor flat no 4 opp ratna ho...,pune mh pune shymala appts 1st floor flat no 4...,False
6,ms consultancy corporation,ms corporation services,shymala appts 1st floor flat no 4 opp ratna ho...,shymala appts 1st floor flat no 4 opp ratna ho...,False
9,team air private limited,teamair com,flat no 101 anuska towers opp mercedes benz sh...,flat no 101 anuska towers opp mercedes benz sh...,False
11,team air private limited,teamair com,flat no 101 anuska towers opp mercedes benz sh...,flat no 101 anuska towers opp mercedes benz sh...,False
12,team air private limited,mirasol,flat no 101 anuska towers opp mercedes benz sh...,flat no 1 101 anuska towers opp mercedes benz ...,False
34,supreme it private limited,स प र म आईट प र इव ट ल म ट ड,office no s 07 82haware centurion plt88 91 sec...,offiec no s 07 82haware centurion plt88 91 sec...,False
36,supreme it private limited,स प र म आईट प र इव ट ल म ट ड,office no s 07 82haware centurion plt88 91 sec...,office no s 07 82haware centurion plt88 91 sec...,False
37,cardiology metro care associates incorporated,cardiologymetrocare com,1138 trimble creek drive west jordan city ut,1138 trimble creek drive west jordan ut,False
38,cardiology metro care associates incorporated,cmcáre com,1138 trimble creek drive west jordan city ut,1138 trimble creek drive west jordan ut,False


In [42]:
def char_ngrams(text, n=3):
    if not text:
        return set()

    text = str(text).replace(" ", "")
    
    if len(text) < n:
        return {text}

    return {
        text[i:i+n]
        for i in range(len(text) - n + 1)
    }


def char_ngram_jaccard(a, b, n=3):
    a_grams = char_ngrams(a, n)
    b_grams = char_ngrams(b, n)

    if not a_grams or not b_grams:
        return 0.0

    return len(a_grams & b_grams) / len(a_grams | b_grams)

hard_token_cases = positive_pairs[
    positive_pairs["shared_tokens_v2"] == 0
].copy()

hard_token_cases["name_char3_jaccard"] = hard_token_cases.apply(
    lambda row: char_ngram_jaccard(
        row["s1_name_for_token"],
        row["matched_name_for_token"],
        n=3
    ),
    axis=1
)

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:

    recall = (
        hard_token_cases["name_char3_jaccard"] >= threshold
    ).mean() * 100

    print(
        f"Name char-3 Jaccard >= {threshold}: "
        f"{recall:.2f}%"
    )

Name char-3 Jaccard >= 0.1: 44.04%
Name char-3 Jaccard >= 0.2: 41.84%
Name char-3 Jaccard >= 0.3: 37.98%
Name char-3 Jaccard >= 0.4: 32.03%
Name char-3 Jaccard >= 0.5: 24.04%


In [43]:
def address_token_overlap(a, b):
    a_tokens = set(str(a).split()) if a else set()
    b_tokens = set(str(b).split()) if b else set()

    if not a_tokens or not b_tokens:
        return False

    return bool(a_tokens & b_tokens)

hard_token_cases["address_token_overlap"] = hard_token_cases.apply(
    lambda row: address_token_overlap(
        row["address_block_s1"],
        row["address_block_matched"]
    ),
    axis=1
)

address_recall = (
    hard_token_cases["address_token_overlap"].mean() * 100
)

print(
    f"Hard cases sharing >=1 address token: "
    f"{address_recall:.2f}%"
)

for threshold in [0.2, 0.3, 0.4, 0.5]:

    char_match = (
        hard_token_cases["name_char3_jaccard"] >= threshold
    )

    address_match = hard_token_cases["address_token_overlap"]

    combined = char_match | address_match

    print(
        f"Char3 >= {threshold} OR address token: "
        f"{combined.mean() * 100:.2f}%"
    )

Hard cases sharing >=1 address token: 99.64%
Char3 >= 0.2 OR address token: 99.93%
Char3 >= 0.3 OR address token: 99.90%
Char3 >= 0.4 OR address token: 99.87%
Char3 >= 0.5 OR address token: 99.80%


In [44]:
print(s2.columns.tolist())
print(s3.columns.tolist())

['entity_id', 'business_name', 'business_address', 'country', 'name_norm', 'block_key']
['entity_id', 'business_name', 'business_address', 'country', 'name_norm', 'block_key']


In [45]:
from collections import Counter

ADDRESS_STOPWORDS = {
    "road",
    "street",
    "avenue",
    "boulevard",
    "drive",
    "lane",
    "highway",
    "parkway",
    "court",
    "place",
    "trail",
    "circle",
    "suite",
    "apartment",
    "floor",
    "flat",
    "unit",
    "no",
    "number",
    "near",
    "opposite",
    "opp",
}

def informative_address_tokens(text):
    if not text:
        return []

    tokens = str(text).split()

    return [
        token
        for token in tokens
        if len(token) >= 2 and token not in ADDRESS_STOPWORDS
    ]

def address_shared_token_count(a, b):
    a_tokens = set(informative_address_tokens(a))
    b_tokens = set(informative_address_tokens(b))

    return len(a_tokens & b_tokens)


positive_pairs["shared_address_tokens"] = positive_pairs.apply(
    lambda row: address_shared_token_count(
        row["address_block_s1"],
        row["address_block_matched"]
    ),
    axis=1
)

for n in [1, 2, 3, 4]:
    recall = (
        positive_pairs["shared_address_tokens"] >= n
    ).mean() * 100

    print(
        f">= {n} shared informative address token(s): "
        f"{recall:.2f}%"
    )

>= 1 shared informative address token(s): 95.57%
>= 2 shared informative address token(s): 94.07%
>= 3 shared informative address token(s): 84.91%
>= 4 shared informative address token(s): 63.87%


In [46]:
address_token_counter = Counter()

for text in positive_pairs["address_block_s1"]:
    address_token_counter.update(
        set(informative_address_tokens(text))
    )

for text in positive_pairs["address_block_matched"]:
    address_token_counter.update(
        set(informative_address_tokens(text))
    )

print("Unique informative address tokens:", len(address_token_counter))

print("\nMost common tokens:")

for token, count in address_token_counter.most_common(40):
    print(f"{token:25} {count}")

Unique informative address tokens: 21674

Most common tokens:
city                      5071
nagar                     4767
maharashtra               4529
new                       4396
delhi                     4068
west                      3457
tx                        3292
pradesh                   2831
mumbai                    2798
ny                        2529
plot                      2496
nc                        2384
north                     2191
oh                        2140
tn                        1885
il                        1808
bangalore                 1694
uttar                     1657
south                     1652
nan                       1615
karnataka                 1513
az                        1478
park                      1476
va                        1470
east                      1459
hyderabad                 1453
colony                    1415
ma                        1411
block                     1397
sector                    1353
nadu    

In [47]:
import pandas as pd

def informative_address_tokens(text):
    if pd.isna(text) or not str(text).strip():
        return []

    tokens = str(text).split()

    return [
        token
        for token in tokens
        if len(token) >= 3 and token not in ADDRESS_STOPWORDS
    ]

ADDRESS_STOPWORDS = {
    # Address-type words
    "road",
    "street",
    "avenue",
    "boulevard",
    "drive",
    "lane",
    "highway",
    "parkway",
    "court",
    "place",
    "trail",
    "circle",
    "suite",
    "apartment",
    "floor",
    "flat",
    "unit",
    "block",
    "plot",
    "sector",
    "nagar",
    "colony",

    # Generic address words
    "near",
    "opposite",
    "opp",
    "main",
    "city",
    "county",
    "north",
    "south",
    "east",
    "west",

    # Very generic fragments
    "the",
    "and",
    "for",
}

hard_token_cases = positive_pairs[
    positive_pairs["shared_tokens_v2"] == 0
].copy()

hard_token_cases["shared_address_tokens"] = hard_token_cases.apply(
    lambda row: len(
        set(informative_address_tokens(row["address_block_s1"])) &
        set(informative_address_tokens(row["address_block_matched"]))
    ),
    axis=1
)

for n in [1, 2, 3]:
    recall = (
        hard_token_cases["shared_address_tokens"] >= n
    ).mean() * 100

    print(
        f"Hard cases with >= {n} informative address tokens: "
        f"{recall:.2f}%"
    )

Hard cases with >= 1 informative address tokens: 99.44%
Hard cases with >= 2 informative address tokens: 94.99%
Hard cases with >= 3 informative address tokens: 80.00%


In [48]:
import pandas as pd

ADDRESS_STOPWORDS = {
    "road",
    "street",
    "avenue",
    "boulevard",
    "drive",
    "lane",
    "highway",
    "parkway",
    "court",
    "place",
    "trail",
    "circle",
    "suite",
    "apartment",
    "floor",
    "flat",
    "unit",
    "block",
    "plot",
    "sector",
    "nagar",
    "colony",
    "near",
    "opposite",
    "opp",
    "main",
    "city",
    "county",
    "north",
    "south",
    "east",
    "west",
    "the",
    "and",
    "for",
}

def informative_address_tokens(text):
    if pd.isna(text) or not str(text).strip():
        return []

    tokens = str(text).split()

    return [
        token
        for token in tokens
        if len(token) >= 3 and token not in ADDRESS_STOPWORDS
    ]

positive_pairs["shared_address_tokens_v2"] = positive_pairs.apply(
    lambda row: len(
        set(informative_address_tokens(row["address_block_s1"])) &
        set(informative_address_tokens(row["address_block_matched"]))
    ),
    axis=1
)

for n in [1, 2, 3]:
    recall = (
        positive_pairs["shared_address_tokens_v2"] >= n
    ).mean() * 100

    print(
        f">= {n} informative address tokens: "
        f"{recall:.2f}%"
    )

>= 1 informative address tokens: 95.43%
>= 2 informative address tokens: 90.87%
>= 3 informative address tokens: 71.90%


In [49]:
from collections import Counter

address_token_frequency = Counter()

for df_name, df in [("S2", s2), ("S3", s3)]:
    print(f"Processing {df_name}...")

    for address, country in zip(
        df["business_address"],
        df["country"]
    ):
        tokens = set(informative_address_tokens(address))

        for token in tokens:
            address_token_frequency[(country, token)] += 1

    print(f"{df_name} done.")

print(
    "Unique country-token blocks:",
    len(address_token_frequency)
)

Processing S2...
S2 done.
Processing S3...
S3 done.
Unique country-token blocks: 2404185


In [50]:
print("\nMost frequent address-token blocks:")

for (country, token), count in address_token_frequency.most_common(40):
    print(
        f"{country:5} | "
        f"{token:25} | "
        f"{count:,}"
    )


Most frequent address-token blocks:
India | ROAD,                     | 310,446
US    | St,                       | 302,807
India | Maharashtra               | 299,122
US    | Rd,                       | 292,618
US    | Street,                   | 291,926
India | Delhi,                    | 287,945
US    | New                       | 273,434
US    | Dr,                       | 269,023
US    | ST,                       | 264,899
US    | Road,                     | 259,245
US    | RD,                       | 258,553
US    | Texas                     | 253,939
India | Road,                     | 244,106
US    | North                     | 242,700
US    | CITY,                     | 240,375
US    | DR,                       | 238,127
US    | Drive,                    | 236,843
US    | STREET,                   | 231,495
India | NO.                       | 229,579
India | FLOOR,                    | 225,286
India | No.                       | 221,956
India | Pradesh                   | 219

In [51]:
import numpy as np

frequencies = np.array(
    list(address_token_frequency.values())
)

print("\nDistribution:")
print("Min:", frequencies.min())
print("Median:", np.median(frequencies))
print("Mean:", frequencies.mean())
print("95th percentile:", np.percentile(frequencies, 95))
print("99th percentile:", np.percentile(frequencies, 99))
print("Max:", frequencies.max())


Distribution:
Min: 1
Median: 2.0
Mean: 26.88178363977814
95th percentile: 30.0
99th percentile: 240.0
Max: 310446


In [52]:
from business_entity_resolution.preprocessing.normalization import normalize_address

In [53]:
import pandas as pd

ADDRESS_STOPWORDS = {
    "road",
    "street",
    "avenue",
    "boulevard",
    "drive",
    "lane",
    "highway",
    "parkway",
    "court",
    "place",
    "trail",
    "circle",
    "suite",
    "apartment",
    "floor",
    "flat",
    "unit",
    "block",
    "plot",
    "sector",
    "nagar",
    "colony",
    "near",
    "opposite",
    "opp",
    "main",
    "city",
    "county",
    "north",
    "south",
    "east",
    "west",
    "the",
    "and",
    "for",
}

def informative_address_tokens(text):
    if pd.isna(text) or not str(text).strip():
        return []

    text = normalize_address(text)

    tokens = text.split()

    return [
        token
        for token in tokens
        if len(token) >= 3 and token not in ADDRESS_STOPWORDS
    ]

In [54]:
positive_pairs["shared_address_tokens_v3"] = positive_pairs.apply(
    lambda row: len(
        set(informative_address_tokens(row["address_block_s1"])) &
        set(informative_address_tokens(row["address_block_matched"]))
    ),
    axis=1
)

for n in [1, 2, 3, 4]:
    recall = (
        positive_pairs["shared_address_tokens_v3"] >= n
    ).mean() * 100

    print(
        f">= {n} informative address tokens: "
        f"{recall:.2f}%"
    )

>= 1 informative address tokens: 95.43%
>= 2 informative address tokens: 90.87%
>= 3 informative address tokens: 71.90%
>= 4 informative address tokens: 42.86%


In [55]:
hard_token_cases = positive_pairs[
    positive_pairs["shared_tokens_v2"] == 0
].copy()

hard_token_cases["shared_address_tokens_v3"] = hard_token_cases.apply(
    lambda row: len(
        set(informative_address_tokens(row["address_block_s1"])) &
        set(informative_address_tokens(row["address_block_matched"]))
    ),
    axis=1
)

for n in [1, 2, 3]:
    recall = (
        hard_token_cases["shared_address_tokens_v3"] >= n
    ).mean() * 100

    print(
        f"Hard cases >= {n} informative address tokens: "
        f"{recall:.2f}%"
    )

Hard cases >= 1 informative address tokens: 99.44%
Hard cases >= 2 informative address tokens: 94.99%
Hard cases >= 3 informative address tokens: 80.00%


In [56]:
address_token_frequency = Counter()

for df_name, df in [("S2", s2), ("S3", s3)]:
    print(f"Processing {df_name}...")

    for address, country in zip(
        df["business_address"],
        df["country"]
    ):
        tokens = set(informative_address_tokens(address))

        for token in tokens:
            address_token_frequency[(country, token)] += 1

    print(f"{df_name} done.")

print(
    "Unique country-token blocks:",
    len(address_token_frequency)
)

Processing S2...
S2 done.
Processing S3...
S3 done.
Unique country-token blocks: 776357


In [57]:
print("\nMost frequent corrected address-token blocks:")

for (country, token), count in address_token_frequency.most_common(40):
    print(
        f"{country:5} | "
        f"{token:25} | "
        f"{count:,}"
    )

frequencies = np.array(
    list(address_token_frequency.values())
)

print("\nDistribution:")
print("Min:", frequencies.min())
print("Median:", np.median(frequencies))
print("Mean:", frequencies.mean())
print("95th percentile:", np.percentile(frequencies, 95))
print("99th percentile:", np.percentile(frequencies, 99))
print("Max:", frequencies.max())


Most frequent corrected address-token blocks:
India | delhi                     | 534,840
India | new                       | 373,306
India | mumbai                    | 360,514
India | maharashtra               | 343,014
US    | new                       | 303,905
US    | texas                     | 294,850
India | pradesh                   | 258,312
India | bangalore                 | 240,821
US    | york                      | 234,238
India | door                      | 221,790
US    | carolina                  | 216,928
US    | ohio                      | 189,217
US    | virginia                  | 177,712
US    | null                      | 174,807
India | pune                      | 171,715
US    | illinois                  | 167,610
India | hyderabad                 | 167,047
India | kolkata                   | 153,726
US    | washington                | 151,460
India | house                     | 139,352
US    | way                       | 138,682
India | uttar                

In [58]:
print(
    s2["business_address"].isna().sum(),
    s3["business_address"].isna().sum()
)

print(
    s2.loc[
        s2["business_address"].astype(str).str.casefold().eq("null"),
        "business_address"
    ].head(10)
)

print(
    s3.loc[
        s3["business_address"].astype(str).str.casefold().eq("null"),
        "business_address"
    ].head(10)
)

168967 175916
Series([], Name: business_address, dtype: str)
Series([], Name: business_address, dtype: str)


In [59]:
ADDRESS_STOPWORDS.add("null")

from itertools import combinations

address_pair_frequency = Counter()

for df_name, df in [("S2", s2), ("S3", s3)]:
    print(f"Processing {df_name}...")

    for address, country in zip(
        df["business_address"],
        df["country"]
    ):
        tokens = sorted(
            set(informative_address_tokens(address))
        )

        for token1, token2 in combinations(tokens, 2):
            address_pair_frequency[
                (country, token1, token2)
            ] += 1

    print(f"{df_name} done.")

print(
    "Unique country-token-pair blocks:",
    len(address_pair_frequency)
)

Processing S2...


MemoryError: 

In [60]:
print("Unique country-token-pair blocks:",
      len(address_pair_frequency))

Unique country-token-pair blocks: 11184810


In [61]:
print("\nMost frequent address-token-pair blocks:")

for (country, token1, token2), count in address_pair_frequency.most_common(30):
    print(
        f"{country:5} | "
        f"{token1:20} | "
        f"{token2:20} | "
        f"{count:,}"
    )


Most frequent address-token-pair blocks:
India | delhi                | new                  | 106,273
India | pradesh              | uttar                | 92,263
India | maharashtra          | mumbai               | 90,982
India | nadu                 | tamil                | 78,837
India | bangalore            | karnataka            | 62,439
India | bengal               | kolkata              | 44,566
India | maharashtra          | pune                 | 44,159
India | andhra               | pradesh              | 40,449
India | chennai              | nadu                 | 31,670
India | chennai              | tamil                | 31,670
India | hyderabad            | telangana            | 30,845
India | madhya               | pradesh              | 28,088
India | maharashtra          | thane                | 21,636
India | ahmedabad            | gujarat              | 20,431
India | bangalore            | cross                | 19,409
India | cross                | karnataka  

In [62]:
from collections import Counter

pair_size_counts = Counter(
    address_pair_frequency.values()
)

print("Unique pair-frequency values:",
      len(pair_size_counts))

print("Min:", min(pair_size_counts))
print("Max:", max(pair_size_counts))

total_pairs = sum(pair_size_counts.values())

print("Total unique pair blocks:", total_pairs)

Unique pair-frequency values: 2100
Min: 1
Max: 106273
Total unique pair blocks: 11184810


In [63]:
sorted_counts = sorted(
    address_pair_frequency.values()
)

def percentile_from_sorted(values, p):
    index = int((len(values) - 1) * p)
    return values[index]

print("Median:",
      percentile_from_sorted(sorted_counts, 0.50))

print("95th percentile:",
      percentile_from_sorted(sorted_counts, 0.95))

print("99th percentile:",
      percentile_from_sorted(sorted_counts, 0.99))

Median: 1
95th percentile: 8
99th percentile: 31


In [64]:
for threshold in [1, 2, 5, 10, 25, 50, 100, 500, 1000]:
    count = sum(
        1 for v in address_pair_frequency.values()
        if v <= threshold
    )

    print(
        f"Pair blocks with frequency <= {threshold}: "
        f"{count:,}"
    )

Pair blocks with frequency <= 1: 5,914,015
Pair blocks with frequency <= 2: 8,355,197
Pair blocks with frequency <= 5: 10,296,445
Pair blocks with frequency <= 10: 10,791,020
Pair blocks with frequency <= 25: 11,046,434
Pair blocks with frequency <= 50: 11,121,812
Pair blocks with frequency <= 100: 11,157,122
Pair blocks with frequency <= 500: 11,180,975
Pair blocks with frequency <= 1000: 11,183,243


In [65]:
def get_country_from_block(block):
    return str(block).split("||", 1)[0]


def get_address_tokens(text):
    return sorted(
        set(informative_address_tokens(text))
    )


def positive_pair_address_block_size(row):
    country = get_country_from_block(row["s1_block_key"])

    tokens = get_address_tokens(
        row["address_block_s1"]
    )

    if len(tokens) < 2:
        return []

    sizes = []

    for token1, token2 in combinations(tokens, 2):
        freq = address_pair_frequency.get(
            (country, token1, token2),
            0
        )

        if freq > 0:
            sizes.append(freq)

    return sizes

In [66]:
positive_pairs["address_pair_freqs"] = positive_pairs.apply(
    positive_pair_address_block_size,
    axis=1
)

In [67]:
for threshold in [1, 2, 5, 10, 25, 50, 100, 500, 1000]:

    captured = positive_pairs["address_pair_freqs"].apply(
        lambda freqs: any(
            freq <= threshold
            for freq in freqs
        )
    )

    print(
        f"At least one address pair with "
        f"frequency <= {threshold}: "
        f"{captured.mean() * 100:.2f}%"
    )

At least one address pair with frequency <= 1: 44.20%
At least one address pair with frequency <= 2: 68.35%
At least one address pair with frequency <= 5: 87.21%
At least one address pair with frequency <= 10: 92.06%
At least one address pair with frequency <= 25: 95.14%
At least one address pair with frequency <= 50: 96.16%
At least one address pair with frequency <= 100: 96.70%
At least one address pair with frequency <= 500: 97.31%
At least one address pair with frequency <= 1000: 97.50%


In [73]:
missed_50=positive_pairs[ ~( positive_pairs["name_token_overlap_v2"] | positive_pairs["block_a_or_b"] 
                             | positive_pairs["address_pair_freqs"].apply(lambda freqs: any(freq<=50 for freq in freqs)))].copy()

print("Missed positive pairs : ",len(missed_50))

Missed positive pairs :  196


In [75]:
missed_50[["source1_entity_id","matched_entity_id","s1_block_key","matched_block_key","address_block_s1","address_block_matched","name_token_overlap_v2",]
    ].head(30)

,source1_entity_id,matched_entity_id,s1_block_key,matched_block_key,address_block_s1,address_block_matched,name_token_overlap_v2
233,S1-786487726,S2-938115929,India||nature building associates,India||lumbrix,29 12 na block 29 shakti nagar block 29 delhi ...,29 12 na block 29 shakti nagar block 29 delhi ...,False
250,S1-571004885,S2-434751720,India||gujarat enterprises private limited,India||ग जर त ए टरप र इज ज प र इव ट ल म ट ड,vardhman plaza lsc sector 6 dwarka plot no 8 f...,plot no 8 f f new delhi south west delhi delhi,False
252,S1-571004885,S3-714238060,India||gujarat enterprises private limited,India||ग जर त ए टरप र इज ज प र इव ट ल म ट ड,vardhman plaza lsc sector 6 dwarka plot no 8 f...,plot no 8 f f vardhman plaza lsc sector 6 dwar...,False
253,S1-571004885,S3-950784712,India||gujarat enterprises private limited,India||ग जर त ए टरप र इज ज प र इव ट ल म ट ड,vardhman plaza lsc sector 6 dwarka plot no 8 f...,plot no 8 f f south west delhi dwarka new delh...,False
625,S1-98096639,S3-16284988,US||carrion equity partners incorporated,US||carrionequitypartners com,tx 3934 lupin bush lane manvel,03934 lupin bush lane manvel texas,False
855,S1-452952997,S3-340176813,US||elysia mendoza metro dte,US||memetro com,1574 colony road fl 0 pasadena md,1574 colony road pasadena maryland,False
1011,S1-627285279,S3-563168432,US||great insurance limited,US||greatinsurance com,13219 annika drive litchfield park az,13219 annika drive litchfield park city arizona,False
1599,S1-635937094,S2-768014389,India||dynamic producer private limited,India||ड यन म क प र ड य सर प र इव ट ल म ट ड,10 ground floor gautam nagar delhi south delhi...,no 10 ground floor gautam nagar delhi south de...,False
1600,S1-635937094,S2-745187312,India||dynamic producer private limited,India||ड यन म क प र ड य सर प र इव ट ल म ट ड,10 ground floor gautam nagar delhi south delhi...,door no 10 ground floor gautam nagar delhi sou...,False
1601,S1-635937094,S2-692166622,India||dynamic producer private limited,India||ड यन म क प र ड य सर प र इव ट ल म ट ड,10 ground floor gautam nagar delhi south delhi...,door no 10 ground floor gautam nagar delhi sou...,False


In [80]:
print(f"Missed:{len(missed_50):,}")

Missed:196


In [82]:
missed_50["num_address_pairs"] = (
    missed_50["address_pair_freqs"]
    .apply(len)
)

missed_50["min_pair_freq"] = (
    missed_50["address_pair_freqs"]
    .apply(lambda freqs: min(freqs) if freqs else float("inf"))
)

print(missed_50["min_pair_freq"].describe())

count    196.00
mean        inf
std         NaN
min       52.00
25%      120.25
50%      778.50
75%         NaN
max         inf
Name: min_pair_freq, dtype: float64


D:\Hackathons\Amazon ML Challenge\V-Max_Hackerz\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4608: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
D:\Hackathons\Amazon ML Challenge\V-Max_Hackerz\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [83]:
print("Missed with min pair freq<=100:",(missed_50["min_pair_freq"]<=100).sum())
print("Missed with min pair freq<=500:",(missed_50["min_pair_freq"]<=500).sum())
print("Missed with min pair freq<=1000:",(missed_50["min_pair_freq"]<=1000).sum())
print("Missed with no address pairs:",(missed_50["num_address_pairs"]==0).sum())

Missed with min pair freq<=100: 42
Missed with min pair freq<=500: 90
Missed with min pair freq<=1000: 109
Missed with no address pairs: 67
